# Reproduce the original Malinois K562 motif catalog

Reproduce the 13 motifs reported for the K562 enhancer library. The paper route uses the original optimized sequences and their frozen Malinois ISM scores. Set `MOTIFSIFT_K562_ARCHIVE` to the root of the K562 reproduction bundle and put MEME Suite 5.5.9 on PATH. A separate optional cell runs fresh optimization with the current software.

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from MotifSift import call_motifs, select_candidates

archive = Path(os.environ.get("MOTIFSIFT_K562_ARCHIVE", "K562_13motifs_reproduction_20261119"))
run = Path(os.environ.get("MOTIFSIFT_K562_RUN", "outputs/malinois_k562_paper"))
run.mkdir(parents=True, exist_ok=True)

## 1. Read the original design library

The 11 optimizers generated 100 candidates each. Exact sequence deduplication gives 1,084 unique enhancers.

In [2]:
designs = pd.read_csv(archive / "inputs/optimization/designs.csv")
with np.load(archive / "inputs/contributions.npz") as data:
    contributions = pd.DataFrame(data["observed"], index=data["sequence_ids"])
designs.groupby("algorithm")["score"].agg(["size", "median", "max"])

,size,median,max
algorithm,,,
adalead,100,25.317344,27.155135
basesearch,100,26.214525,27.832603
beam_search,100,27.427982,30.767609
cross_entropy_method,100,28.943090,29.598214
den,100,23.908835,25.174662
fast_seqprop,100,26.637044,28.835720
genetic_algorithm,100,33.412852,34.887356
ism_greedy,100,28.653624,31.538803
ledidi,100,24.768156,26.851628


## 2. Discover motifs

Run the original single-pass discovery from the archived contributions. Use all positive seqlets, the seed-42 dinucleotide background, XSTREME and the native FIMO support and redundancy filters.

In [3]:
motifs = call_motifs(designs, contributions=contributions, output_dir=run / "motifs")
expected = pd.read_csv(archive / "expected/motif_sites.csv")
actual = pd.read_csv(run / "motifs/motif_sites.csv")
order = ["motif_id", "sequence_name", "start0", "end0", "strand"]
pd.testing.assert_frame_equal(actual.sort_values(order).reset_index(drop=True),
                              expected.sort_values(order).reset_index(drop=True), check_exact=True)
assert len(motifs) == 13 and len(actual) == 9464
motifs.support.query("retained == True")[["motif_id", "seqlets", "parents", "sites_on_unique_DNA"]]

,motif_id,seqlets,parents,sites_on_unique_DNA
0,1-VAGATAAGG,3098,1057,5342.0
1,2-AMCAKMTGKT,449,377,1440.0
2,3-CTGAGTCAYC,332,269,992.0
3,4-ATGCCTTCCG,50,50,103.0
4,5-SGCTATCAGSC,353,327,123.0
5,6-CCACACC,31,31,390.0
6,7-CGAGAATTGA,48,48,110.0
7,8-TTTCCWGGAAA,50,42,543.0
8,9-AATCATCCC,82,80,44.0
9,10-CAGGAAACG,115,99,62.0


## 3. Select a panel

Use the original catalog and retain candidates above the 20th-percentile specificity cutoff for selection.

In [4]:
gate = float(designs.drop_duplicates("sequence").score.quantile(0.2))
panel = select_candidates(designs, motifs=motifs, top_n=10, min_score=gate, output_dir=run / "selected")
panel[["selection_rank", "sequence", "score", "present_motifs"]]

,selection_rank,sequence,score,present_motifs
0,1,GTTATCAAACCGCTGTGTCTGGTTCGTACAGATAACGCTGACATTT...,34.887356,1-VAGATAAGG;2-AMCAKMTGKT;3-CTGAGTCAYC;8-TTTCCW...
254,2,GATGTGGCGGTTAGATAAGTTAAATGTGGCTGTCAACGATAGGCCT...,28.800381,1-VAGATAAGG;6-CCACACC
390,3,GGGAGTCTCCTTATCAGTGGGAACACATGTTGCCGAGTCACTCATG...,27.099779,1-VAGATAAGG;2-AMCAKMTGKT;3-CTGAGTCAYC;4-ATGCCT...
128,4,TATCTTCGGGGAACATCTGGTGACCAGAGATAAGGCGTTTACCCGT...,29.598214,1-VAGATAAGG;2-AMCAKMTGKT;5-SGCTATCAGSC;22-GGCA...
304,5,TCTTGCGCGAGGGTGCCTTATCTGTTCCTTGCAGCACTGTGGGGTT...,28.136970,1-VAGATAAGG;4-ATGCCTTCCG;8-TTTCCWGGAAA
797,6,TGGCGCTCCTTATCGGGCGACACGCTAAAAATAACTGACGGTCAGA...,24.568838,1-VAGATAAGG;3-CTGAGTCAYC;18-TCATGACT
490,7,TCCGTCCTAACTGATCAGCAGATATATATATAGAGTTCCAGGAAAC...,26.464320,1-VAGATAAGG;3-CTGAGTCAYC;6-CCACACC;8-TTTCCWGGA...
823,8,TTGATAACCGTTTTTGGCCACTCCTGCCTGCTGTTTCCTGATAACA...,24.342896,1-VAGATAAGG;2-AMCAKMTGKT;6-CCACACC;8-TTTCCWGGA...
569,9,TCTCTCACAGTGACCCACACCTCCCGAACCAGAAGTCGGCCTTATC...,26.064612,1-VAGATAAGG;2-AMCAKMTGKT;3-CTGAGTCAYC;5-SGCTAT...
447,10,GTTAGAGATAAGGCTTATCTCTTATCTTCAACAGGCAGCTGTTGGG...,26.641018,1-VAGATAAGG;2-AMCAKMTGKT;11-MAACAGGCAR


## Optional. Run a new design

Set `RUN_FRESH = True` to design and interpret a new enhancer library in FP32 with seed 42 and fixed prediction batches. This uses the current optimization engine. The historical candidate library was generated with parallel GPU scoring, so a fresh run can return a different library and motif catalog.

In [5]:
RUN_FRESH = False

if RUN_FRESH:
    # %pip install "bioseq-models @ git+https://github.com/UTR-CODE/bioseq-models.git@84a5075c3656127d99027c38705212a2de6e62c9"
    import os
    from pathlib import Path

    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

    import numpy as np
    import torch
    from bioseq_models import load_model
    from bioseq_models.adapters.malinois import LEFT_CONTEXT, RIGHT_CONTEXT
    from MotifSift import call_motifs, random_seed_sequences, select_candidates
    from sequence_optimization import MultiTaskObjective, SequencePredictor, onehot
    from sequence_optimization.api import as_score_batch
    from sequence_optimization.suite import generate_designs

    torch.set_num_threads(1)
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.use_deterministic_algorithms(True)

    CELLS = ("K562", "HepG2", "SKNSH")

    class ContextPredictor(SequencePredictor):
        def prepare_input(self, values):
            left = self.left.to(values).expand(len(values), -1, -1)
            right = self.right.to(values).expand(len(values), -1, -1)
            return torch.cat((left, values, right), dim=1).transpose(1, 2)

    class SpecificityScorer:
        supports_soft_scoring = True

        def __init__(self, raw, target):
            self.raw = raw
            self.target = CELLS.index(target)
            self.off = [i for i in range(3) if i != self.target]
            self.task_names = (*CELLS, *(f'{target}_minus_{CELLS[i]}' for i in self.off))
            self.device = 'cuda'
            self.soft = ContextPredictor(raw.model, device='cuda', tasks=CELLS, batch_size=1024)
            self.soft.left = torch.from_numpy(onehot([LEFT_CONTEXT], channels_first=False))
            self.soft.right = torch.from_numpy(onehot([RIGHT_CONTEXT], channels_first=False))

        def score(self, sequences):
            x = as_score_batch(self.raw.score(sequences)).scores
            return np.column_stack([x, *(x[:, self.target] - x[:, i] for i in self.off)])

        def score_soft(self, probabilities):
            x, _ = self.soft.score_soft(probabilities)
            return torch.cat([x, *(x[:, self.target:self.target+1] - x[:, i:i+1]
                                   for i in self.off)], dim=1), None

    model = SpecificityScorer(load_model("malinois", device="cuda", batch_size=1024), "K562")
    objective = MultiTaskObjective(high=[3, 4], aggregation="worst")
    fresh = Path("outputs/malinois_k562_fresh")
    fresh.mkdir(parents=True, exist_ok=True)
    seeds = random_seed_sequences(200, 100, seed=42)
    fresh_designs = generate_designs(
        model, length=200, initial=seeds, objective=objective,
        seed=42, output_dir=fresh / "designs",
    )
    fresh_motifs = call_motifs(fresh_designs, model, objective=objective, output_dir=fresh / "motifs")
    fresh_gate = float(fresh_designs.drop_duplicates("sequence").score.quantile(0.2))
    select_candidates(fresh_designs, motifs=fresh_motifs, top_n=10,
                      min_score=fresh_gate, output_dir=fresh / "selected")
    print(f"{len(fresh_motifs)} final motifs in the fresh library")